In [ ]:
pip install faker pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 53.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker('id_ID')

# Konfigurasi dasar
n_data = 10000
region_to_cities = {
    'Sumatra': ['Medan', 'Palembang', 'Pekanbaru', 'Bandar Lampung', 'Padang', 'Jambi'],
    'Jawa': ['Jakarta', 'Surabaya', 'Bandung', 'Semarang', 'Bekasi', 'Depok', 'Tangerang', 'Yogyakarta', 'Malang'],
    'Kalimantan': ['Balikpapan', 'Banjarmasin', 'Pontianak', 'Samarinda', 'Palangkaraya'],
    'Sulawesi': ['Makassar', 'Manado', 'Palu', 'Kendari', 'Gorontalo'],
    'Bali & Nusa Tenggara': ['Denpasar', 'Mataram', 'Kupang']
}

city_to_region = {city: region for region, cities in region_to_cities.items() for city in cities}

# Koordinat nyata untuk masing-masing kota (disederhanakan, bisa ditambah/ubah sesuai kebutuhan)
city_coords = {
    'Medan': (3.5952, 98.6722), 'Palembang': (-2.9909, 104.7566), 'Pekanbaru': (0.5071, 101.4478),
    'Bandar Lampung': (-5.3971, 105.2668), 'Padang': (-0.9471, 100.4172), 'Jambi': (-1.4852, 102.4381),
    'Jakarta': (-6.2088, 106.8456), 'Surabaya': (-7.2504, 112.7688), 'Bandung': (-6.9175, 107.6191),
    'Semarang': (-6.9667, 110.4167), 'Bekasi': (-6.2416, 106.9924), 'Depok': (-6.4025, 106.7942),
    'Tangerang': (-6.1702, 106.6415), 'Yogyakarta': (-7.7956, 110.3695), 'Malang': (-7.9797, 112.6304),
    'Balikpapan': (-1.2654, 116.8312), 'Banjarmasin': (-3.3201, 114.5908), 'Pontianak': (0.0263, 109.3425),
    'Samarinda': (-0.5022, 117.1536), 'Palangkaraya': (-2.2096, 113.9145),
    'Makassar': (-5.1477, 119.4327), 'Manado': (1.4748, 124.8421), 'Palu': (-0.8917, 119.8707),
    'Kendari': (-3.9766, 122.5151), 'Gorontalo': (0.5372, 123.0615),
    'Denpasar': (-8.6500, 115.2167), 'Mataram': (-8.5833, 116.1167), 'Kupang': (-10.1772, 123.6070),
}

company = ['SIG', 'SGI']
order_types = ['SO', 'TO']
credit_hold_status = ['No Hold', 'Credit Hold', 'Aging Hold']
gas_types = ['OXYGEN', 'NITROGEN', 'HYDROGEN', 'ARGON', 'CARBON DIOXIDE']
status_mapping = {
    None: '0 New',
    'Order Held': 'Order Held',
    'Open Order': '1. Open Order',
    'Credit/Aging Hold': '2. Credit/Aging Hold',
    'Order Released': '3. Order Released',
    'Allocated': '4. Allocated',
    'Ready to Deliver': '5. Ready to Deliver',
    'Invoiced - Not Posted': '6. Shipped',
    'Invoiced - Posted': '6. Shipped',
    'Order Closed': '6. Shipped',
    'Order Closed - Shipped': '6. Shipped',
    'Shipped': '6. Shipped',
    'Order Closed - Not Shipped': '7. Closed'
}

# Nama entry person dan created user yang berhubungan
persons = [fake.first_name() for _ in range(15)]
created_users = [f"EMP{str(100000+i)}" for i in range(15)]
entry_user_map = dict(zip(persons, created_users))

def random_entry_and_user():
    name = random.choice(persons)
    return name, entry_user_map[name]

data = []

for i in range(n_data):
    comp = random.choice(company)
    plant_id = f"{comp}{str(random.randint(1, 20)).zfill(3)}"
    cities = list(city_to_region.keys())
    city = random.choice(cities)
    region = city_to_region[city]

    order_date = fake.date_between(
        start_date=datetime(2025, 1, 1).date(),
        end_date=datetime(2025, 12, 31).date()
    )
    ship_date = order_date + timedelta(days=random.randint(1, min(100, 10)))  # biasanya 1-7 hari
    order_type = random.choice(order_types)
    order_num = 500000 + i
    cust_id = random.randint(100000, 999999)
    credit_hold = random.choice(credit_hold_status)
    part_num = f"CYOXGFMR{str(random.randint(1,999999)).zfill(7)}"
    sell_qty = random.randint(1, 100)
    pack_num = random.randint(100000, 999999)
    trip_num = f"{plant_id}-{random.randint(100000, 999999)}"
    nopol = f"{random.choice(['B', 'D', 'L', 'AB'])} {random.randint(1000,9999)} {random.choice(['XY', 'YZ', 'ZZ'])}"
    driver = fake.name()

    shiptonum = random.choice(['', str(random.randint(1,10))])
    shiptoname = fake.company() if shiptonum else ''
    shiptocity = random.choice(cities) if shiptonum else ''

    # Ambil koordinat dari mapping kota
    plant_coord = city_coords[city]
    cust_coord = city_coords[shiptocity] if shiptonum and shiptocity in city_coords else (
        round(random.uniform(-8.5, 1.5), 6), round(random.uniform(106.0, 120.0), 6)
    )

    distance = round(np.linalg.norm(np.array(cust_coord) - np.array(plant_coord)), 6) * 111
    np_ll = (round(random.uniform(-8.5, 1.5), 6), round(random.uniform(106.0, 120.0), 6))
    np_site_id = f"{comp}{str(random.randint(1, 20)).zfill(3)}"
    np_site = random.choice(cities)
    distance_nearest = round(np.linalg.norm(np.array(cust_coord) - np.array(np_ll)), 6) * 111
    gap = distance - distance_nearest

    entry_person, created_user = random_entry_and_user()

    # Status dan SCM Mapping
    status = random.choices(
        population=[
            'Shipped', 'Ready to Deliver', 'Allocated', 'Order Released',
            'Credit/Aging Hold', 'Open Order', 'Order Closed - Not Shipped'
        ],
        weights=[40, 15, 10, 10, 5, 10, 10],
        k=1
    )[0]
    status_scm = status_mapping.get(status)

    data.append([
        comp, plant_id, plant_id, city, region, order_date.strftime('%d/%m/%Y'), order_type,
        order_num, 1, 1, cust_id, fake.company(), credit_hold, part_num, sell_qty, pack_num,
        ship_date.strftime('%d/%m/%Y'), trip_num, nopol, driver, shiptonum, shiptoname, shiptocity,
        f"{plant_coord[0]}, {plant_coord[1]}", f"{cust_coord[0]}, {cust_coord[1]}", distance,
        f"{np_ll[0]}, {np_ll[1]}", np_site_id, np_site, distance_nearest, gap,
        entry_person, created_user, status, status_scm
    ])

columns = [
    'Company', 'Order Plant', 'Release Plant', 'Site Name', 'Region', 'OrderDate', 'OrderType',
    'OrderNum', 'OrderLine', 'OrderRelNum', 'CustID', 'Name', 'CreditHold', 'PartNum',
    'SellingReqQty', 'PackNum', 'ShipDate', 'TripNum', 'Nopol', 'Driver', 'ShipToNum',
    'ShipToName', 'ShipToCity', 'Plant Coordinate', 'Cust. Coordinate', 'Distance KM',
    'NP LL', 'NP SiteID', 'NP Site', 'Distance KM Nearest', 'NP Gap', 'EntryPerson',
    'Created User', 'Status', 'Status SCM'
]

df = pd.DataFrame(data, columns=columns)

# Simpan ke CSV jika perlu
# df.to_csv('dummy_data_sgi_cleaned.csv', index=False)


In [ ]:
import gspread
from gspread_dataframe import set_with_dataframe
from oauth2client.service_account import ServiceAccountCredentials
import pandas as pd

# 1. Scope
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]

# 2. Autentikasi
creds = ServiceAccountCredentials.from_json_keyfile_name("credentials.json", scope)
client = gspread.authorize(creds)

# 3. Buka spreadsheet dan worksheet (pastikan nama sesuai)
spreadsheet = client.open("SCM Replica Data")  # Ganti dengan nama spreadsheet kamu
sheet = spreadsheet.sheet1  # Atau .worksheet("Nama Sheet")

# 4. Upload DataFrame ke Google Sheets
set_with_dataframe(sheet, df)  # df adalah DataFrame kamu